# Práctica 2 — Dynamic Matrix Control (DMC)

En esta práctica se implementa explícitamente el bucle de control predictivo.

Se estudiarán:

1. el modelo basado en respuesta escalón;
2. el truncamiento mediante $N_t$;
3. la construcción de las matrices $G$ y $F$;
4. la separación entre respuesta libre y respuesta forzada;
5. la función objetivo que penaliza $\Delta u$;
6. la ley de control DMC sin restricciones;
7. el horizonte deslizante;
8. el efecto de $h$, $\lambda$ y $N_t$;
9. la diferencia fundamental entre MAC y DMC respecto al error estacionario.



> **Dinámica recomendada:** antes de ejecutar cada experimento, analice la pregunta asociada y realice una predicción cualitativa. Después, ejecute la celda, compare el resultado con la predicción y despliegue la respuesta razonada.  
> Las respuestas están ocultas en bloques desplegables para poder relizar un análisis pausado.

## Objetivos

Al finalizar la práctica se debería ser capaz de:

- obtener un modelo discreto de una planta continua;
- obtener los coeficientes $g_i$ de un modelo de respuesta escalón;
- comprender el efecto del truncamiento $N_t$;
- construir el predictor DMC

\begin{align*}
\hat{\mathbf y}=G\Delta\mathbf u^+ + \mathbf f;
\end{align*}

- calcular la respuesta libre mediante

\begin{align*}
\mathbf f=F\Delta\mathbf u^-+\mathbf 1\,y(k);
\end{align*}

- obtener la ley de control DMC sin restricciones;
- implementar el principio de horizonte deslizante;
- analizar el efecto de $h$, $\lambda$ y $N_t$;
- comparar el comportamiento estacionario de MAC y DMC.

## 1. Librerías

Se utilizan únicamente `NumPy`, `SciPy` y `Matplotlib`, disponibles de forma estándar en Google Colab.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

np.set_printoptions(precision=5, suppress=True)

## 2. Descripción del proceso

El proceso es el control de nivel de un tanque.

![My Image](https://github.com/GICI-UPV-EHU/Imagenes_GICI/blob/main/Procesos/Tanque.png?raw=true)

El balance de caudales es

\begin{align*}
A_t\frac{dh(t)}{dt}=q(t)-q_1(t),
\end{align*}

y, suponiendo flujo laminar,

\begin{align*}
q_1(t)=K_t h(t).
\end{align*}

Por tanto,

\begin{align*}
A_t\frac{dh(t)}{dt}+K_t h(t)=q(t).
\end{align*}

La función de transferencia entre el caudal de entrada y la altura es

\begin{align*}
G(s)
&=\frac{H(s)}{Q(s)} \\
&=\frac{1}{A_t s+K_t}.
\end{align*}

Se establecen unos parámetros del tanque,

\begin{align*}
A_t&=1, \\
K_t&=0.5,
\end{align*}

se obtiene

\begin{align*}
\boxed{G(s)=\frac{1}{s+0.5}}.
\end{align*}

In [ ]:
# Parámetros del tanque
At = 1.0
Kt = 0.5

# G(s) = 1 / (At*s + Kt)
G_cont = signal.TransferFunction([1.0], [At, Kt])

print("Modelo continuo:")
print(G_cont)

### Preguntas para dinamizar — interpretación física del proceso

> **Pregunta 1.** Sin hacer ningún cálculo, ¿qué ocurrirá con la altura si mantenemos constante el caudal de entrada? ¿Crecerá indefinidamente?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. El caudal de salida aumenta con el nivel según $q_1(t)=K_t h(t)$. El equilibrio se alcanza cuando

\begin{align*}
q=q_1=K_t h.
\end{align*}

Por tanto,

\begin{align*}
h_{\mathrm{eq}}=\frac{q}{K_t}.
\end{align*}

Con $K_t=0.5$, una entrada constante $q=1$ conduce a $h_{\mathrm{eq}}=2$.

</details>

> **Pregunta 2.** Si duplicamos el área $A_t$ manteniendo $K_t$, ¿cambia la altura final? ¿Y la velocidad de respuesta?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

La altura estacionaria no cambia porque

\begin{align*}
G(0)=\frac{1}{K_t}.
\end{align*}

Sin embargo, la constante de tiempo

\begin{align*}
\tau=\frac{A_t}{K_t}
\end{align*}

se duplica. El tanque responde más lentamente.

</details>

> **Pregunta 3.** ¿Qué sucede si aumentamos $K_t$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Disminuyen tanto la ganancia estática como la constante de tiempo:

\begin{align*}
K=\frac{1}{K_t},
\qquad
\tau=\frac{A_t}{K_t}.
\end{align*}

Por tanto, ante el mismo caudal el tanque alcanza una altura menor y lo hace más rápidamente.

</details>

## 3. Discretización

Se utiliza un periodo de muestreo de $T_d=0.2$.

La discretización se realizará mediante retenedor de orden cero (ZOH). Para la simulación en lazo cerrado usaremos la representación

\begin{align*}
x(k+1)&=A_d x(k)+B_d u(k), \\
y(k)&=C_d x(k)+D_d u(k).
\end{align*}

Que es la representación en espacio de estados, pero se emplea simplemente por comodidad de posteriores desarrollos.

In [ ]:
Td = 0.2

A = np.array([[-Kt / At]])
B = np.array([[1.0 / At]])
C = np.array([[1.0]])
D = np.array([[0.0]])

Ad, Bd, Cd, Dd, _ = signal.cont2discrete(
    (A, B, C, D), Td, method="zoh"
)

Ad = float(Ad[0, 0])
Bd = float(Bd[0, 0])
Cd = float(Cd[0, 0])
Dd = float(Dd[0, 0])

print(f"Td = {Td}")
print(f"Ad = {Ad:.8f}")
print(f"Bd = {Bd:.8f}")
print(f"Cd = {Cd:.8f}")
print(f"Dd = {Dd:.8f}")
print()
print("Modelo discreto:")
print(f"h(k+1) = {Ad:.8f} h(k) + {Bd:.8f} q(k)")

### Preguntas para dinamizar — periodo de muestreo

Para esta planta,

\begin{align*}
\tau=\frac{A_t}{K_t}=2.
\end{align*}

Con $T_d=0.2$ se obtienen aproximadamente 10 muestras por constante de tiempo.

> **Pregunta 4.** ¿Os parece $T_d=0.2$ un valor razonable antes de realizar ninguna simulación?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Sí, como criterio práctico para esta planta:

\begin{align*}
\frac{\tau}{T_d}=\frac{2}{0.2}=10.
\end{align*}

La dinámica dominante queda suficientemente muestreada sin aumentar innecesariamente la carga computacional.

</details>

> **Pregunta 5.** ¿Qué esperáis que ocurra si cambiamos $T_d=0.2$ por $T_d=1$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Tendremos muchas menos muestras para describir la misma dinámica. Para este sistema,

\begin{align*}
A_d=e^{-T_d/\tau}.
\end{align*}

Así,

\begin{align*}
T_d=0.2 &\Rightarrow A_d\simeq0.9048,\\
T_d=1.0 &\Rightarrow A_d\simeq0.6065.
\end{align*}

Cada paso discreto representa una evolución mucho mayor del proceso.

</details>

> **Pregunta 6.** Si la discretización ZOH es exacta en los instantes de muestreo, ¿por qué no podemos elegir un $T_d$ arbitrariamente grande?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque exactitud de discretización no equivale a frecuencia de control adecuada. Con un $T_d$ grande:

- medimos con menos frecuencia;
- actuamos con menos frecuencia;
- perdemos resolución temporal;
- el DMC actualiza sus decisiones más lentamente.

</details>

In [ ]:
# Experimento en directo: efecto del periodo de muestreo
# Antes de ejecutar, intentad predecir cómo cambiarán Ad y Bd.

tau = At / Kt

for Td_test in [0.05, 0.2, 0.5, 1.0, 2.0]:
    Ad_t, Bd_t, _, _, _ = signal.cont2discrete(
        (A, B, C, D), Td_test, method="zoh"
    )
    print(
        f"Td={Td_test:4.2f} s   "
        f"muestras/tau={tau/Td_test:5.1f}   "
        f"Ad={Ad_t[0,0]:.4f}   "
        f"Bd={Bd_t[0,0]:.4f}"
    )

> **Pregunta 7.** ¿Por qué $A_d$ se aproxima a 1 cuando $T_d$ disminuye?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque en un intervalo muy corto el estado cambia poco. Matemáticamente,

\begin{align*}
\lim_{T_d\to 0} e^{-T_d/\tau}=1.
\end{align*}

</details>

> **Pregunta 8.** DMC no utilizará directamente $A_d$ ni $B_d$. ¿Para qué hemos discretizado la planta?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Para obtener su respuesta escalón discreta

\begin{align*}
g_1,g_2,\ldots,g_{N_t},
\end{align*}

que será el modelo que DMC utilizará directamente en el predictor.

</details>

## 4. Respuesta escalón del sistema

DMC utiliza como modelo la secuencia de coeficientes de respuesta escalón $g_i$.

Para un sistema estable, el modelo ideal puede escribirse como

\begin{align*}
y(k)=\sum_{i=1}^{\infty}g_i\,\Delta u(k-i),
\end{align*}

donde

\begin{align*}
\Delta u(k)=u(k)-u(k-1).
\end{align*}

En la implementación real se trunca la secuencia:

\begin{align*}
y(k)\approx\sum_{i=1}^{N_t}g_i\,\Delta u(k-i).
\end{align*}

La siguiente función reproduce la salida del sistema ante una entrada escalón.

In [ ]:
def discrete_step_response(Ad, Bd, Cd, Dd, n_steps):
    """Respuesta de un sistema discreto SISO de primer orden ante u(k)=1."""
    y = np.zeros(n_steps + 1)
    x = 0.0

    for k in range(n_steps):
        uk = 1.0
        y[k] = Cd * x + Dd * uk
        x = Ad * x + Bd * uk

    y[n_steps] = Cd * x + Dd
    return y


tfin = 20.0
n_step = int(round(tfin / Td))
t_disc = np.arange(n_step + 1) * Td

ystep = discrete_step_response(Ad, Bd, Cd, Dd, n_step)

print("Primeras muestras [y(0), g1, g2, ...]:")
print(ystep[:8])

### Preguntas para dinamizar — significado de los coeficientes $g_i$

> **Pregunta 9.** ¿Qué representa físicamente cada coeficiente $g_i$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

$g_i$ es el valor de la salida $i$ muestras después de aplicar un escalón unitario a la entrada. La secuencia describe cómo se acumula en el tiempo el efecto de un cambio permanente de entrada.

</details>

> **Pregunta 10.** ¿Por qué en DMC aparecen incrementos $\Delta u$ si el modelo se obtiene con una respuesta escalón?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque cualquier secuencia de entrada puede interpretarse como una suma de escalones incrementales. Cada cambio

\begin{align*}
\Delta u(k)=u(k)-u(k-1)
\end{align*}

genera una respuesta escalón desplazada, y la salida total se obtiene por superposición.

</details>

> **Pregunta 11.** Para esta planta estable, ¿qué comportamiento esperáis en la secuencia $g_i$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Debe crecer desde cero y aproximarse asintóticamente a la ganancia estática:

\begin{align*}
g_i\rightarrow g_\infty=G(0)=2.
\end{align*}

</details>

> **Pregunta 12.** ¿Qué relación existe entre los coeficientes de respuesta escalón $g_i$ y los coeficientes impulsionales $h_i$ utilizados en MAC?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Se cumple

\begin{align*}
h_i=g_i-g_{i-1},
\end{align*}

tomando $g_0=0$. La respuesta impulsional es la diferencia entre dos respuestas escalón desplazadas una muestra.

</details>

In [ ]:
# Experimento en directo: relación entre modelo escalón e impulsional
# h_i = g_i - g_{i-1}

g_demo = ystep[:11]
h_from_g = np.diff(g_demo)

print(" i        g_i          h_i = g_i-g_(i-1)")
print("------------------------------------------")
for i in range(1, len(g_demo)):
    print(f"{i:2d}    {g_demo[i]:10.7f}      {h_from_g[i-1]:10.7f}")

> **Pregunta 13.** ¿Qué ocurre con $h_i=g_i-g_{i-1}$ cuando $g_i$ se aproxima a su valor estacionario?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Los incrementos entre muestras se hacen cada vez menores y, por tanto,

\begin{align*}
h_i\rightarrow 0.
\end{align*}

Esta es la conexión entre el truncamiento impulsional de MAC y el truncamiento escalón de DMC.

</details>

> **Pregunta 14.** En la comparación continuo–discreto, ¿coincidir en los instantes de muestreo significa que ambos modelos son iguales en todo instante?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. El modelo discreto representa solo los valores en $kT_d$. El sistema continuo sigue evolucionando entre muestras.

</details>

### Comparación entre modelo continuo y modelo discreto

La discretización ZOH permite comparar directamente la respuesta continua y sus muestras discretas.

In [ ]:
t_cont = np.linspace(0.0, tfin, 5001)
u_cont = np.ones_like(t_cont)

tout, y_cont, _ = signal.lsim(G_cont, U=u_cont, T=t_cont)

plt.figure(figsize=(9, 4))
plt.plot(tout, y_cont, label="Modelo continuo")
plt.step(t_disc, ystep, where="post", label="Modelo discreto")
plt.xlabel("Tiempo")
plt.ylabel("Salida")
plt.title("Respuesta escalón")
plt.grid(True)
plt.legend()
plt.show()

## 5. Elección del truncamiento $N_t$

En DMC el truncamiento no se decide porque los coeficientes $g_i$ se hagan cero, sino porque hayan alcanzado prácticamente su valor estacionario.

Para esta planta,

\begin{align*}
g_{\infty}
&=G(0) \\
&=\frac{1}{K_t} \\
&=2.
\end{align*}

Definiremos un error relativo de truncamiento como

\begin{align*}
\varepsilon_t(N_t)
=
\frac{\left|g_{\infty}-g_{N_t}\right|}
     {\left|g_{\infty}\right|}.
\end{align*}

En la siguiente celda se cuantifica explícitamente el error de truncamiento para los siguientes valores $N_t = 10, 20, 40, 60, 80$.

In [ ]:
g_full = ystep[1:]  # se elimina y(0)=0
g_inf = 1.0 / Kt

def truncation_error_step(g_full, Nt, g_inf):
    return abs(g_inf - g_full[Nt - 1]) / abs(g_inf)

for Nt_test in [10, 20, 40, 60, 80]:
    print(
        f"Nt={Nt_test:2d} -> error de truncamiento = "
        f"{100*truncation_error_step(g_full, Nt_test, g_inf):7.4f} %"
    )

### Preguntas para dinamizar — elección de $N_t$

> **Pregunta 15.** ¿Por qué en DMC no esperamos que $g_i$ llegue a cero?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque $g_i$ pertenece a una respuesta escalón. En una planta estable converge al valor estacionario

\begin{align*}
g_\infty=G(0),
\end{align*}

no a cero.

</details>

> **Pregunta 16.** ¿Qué significa físicamente la aproximación $g_i\simeq g_{N_t}$ para $i>N_t$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Que a partir de $N_t$ consideramos que el efecto de un escalón ya ha alcanzado prácticamente su régimen estacionario y no cambia de forma relevante.

</details>

> **Pregunta 17.** ¿Qué ocurre si elegimos $N_t$ demasiado pequeño?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El controlador supone prematuramente que la respuesta ya se ha estabilizado. Introduce así un error sistemático en la respuesta libre y en las predicciones futuras.

</details>

> **Pregunta 18.** ¿Tiene sentido aumentar $N_t$ indefinidamente?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Una vez que $g_{N_t}$ está suficientemente cerca de $g_\infty$, aumentar $N_t$ apenas mejora el modelo y sí incrementa memoria y cálculo.

</details>

### Modelo escalón truncado

Cuando se alcanza $N_t$, DMC supone que la respuesta ya se encuentra en régimen estacionario. Por tanto, para índices posteriores se toma

\begin{align*}
g_i \approx g_{N_t},
\qquad i>N_t.
\end{align*}

La siguiente función compara el modelo truncado con la respuesta real del sistema discreto.

In [ ]:
def truncated_step_model(gi, n_samples):
    """Respuesta escalón del modelo DMC truncado, manteniendo g_Nt después de Nt."""
    gi = np.asarray(gi, dtype=float)
    Nt = len(gi)

    y = np.zeros(n_samples + 1)
    for k in range(1, n_samples + 1):
        if k <= Nt:
            y[k] = gi[k - 1]
        else:
            y[k] = gi[-1]
    return y


Nt = 60
gi = g_full[:Nt]

y_trunc = truncated_step_model(gi, n_step)

plt.figure(figsize=(9, 4))
plt.step(t_disc, ystep, where="post", label="Modelo discreto")
plt.step(t_disc, y_trunc, where="post", label=f"Modelo DMC, Nt={Nt}")
plt.xlabel("Tiempo")
plt.ylabel("Salida")
plt.title("Efecto del truncamiento del modelo escalón")
plt.grid(True)
plt.legend()
plt.show()

print(
    f"Error relativo en g_Nt para Nt={Nt}: "
    f"{100*truncation_error_step(g_full, Nt, g_inf):.4f} %"
)

In [ ]:
# Experimento en directo: comparar varios truncamientos del modelo escalón

Nt_demo = [10, 20, 40, 60, 80]

plt.figure(figsize=(9, 4))
plt.step(t_disc, ystep, where="post", linewidth=2, label="Modelo discreto")

for Nt_test in Nt_demo:
    gi_test = g_full[:Nt_test]
    y_test = truncated_step_model(gi_test, n_step)
    plt.step(t_disc, y_test, where="post", label=f"Nt={Nt_test}", alpha=0.8)

plt.xlabel("Tiempo")
plt.ylabel("Salida")
plt.title("Comparación de distintos truncamientos DMC")
plt.grid(True)
plt.legend()
plt.show()

print("Error de truncamiento:")
for Nt_test in Nt_demo:
    print(
        f"Nt={Nt_test:2d} -> "
        f"{100*truncation_error_step(g_full, Nt_test, g_inf):8.4f} %"
    )

> **Pregunta 19.** Si reducimos $T_d$ a la mitad y mantenemos el mismo $N_t$, ¿representamos la misma duración física de la dinámica?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. La ventana temporal aproximada es

\begin{align*}
T_{\mathrm{mem}}\simeq N_t T_d.
\end{align*}

Al reducir $T_d$ a la mitad, la misma cantidad de coeficientes cubre la mitad de tiempo físico.

</details>

> **Pregunta 20.** ¿Por qué el modelo escalón DMC resulta problemático para una planta inestable?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque la respuesta escalón no converge a un valor estacionario finito. No existe entonces un $g_{N_t}$ que pueda representar razonablemente la cola de la respuesta.

</details>

## 6. Predictor vectorial DMC

En el instante $k$, la predicción a $j$ pasos puede expresarse como la suma de:

- una **respuesta forzada**, asociada a los futuros incrementos de control;
- una **respuesta libre**, asociada a la salida actual y a los incrementos de control pasados.

El predictor vectorial queda

\begin{align*}
\hat{\mathbf y}
=
G\Delta\mathbf u^+
+
\mathbf f.
\end{align*}

El vector de incrementos futuros es

\begin{align*}
\Delta\mathbf u^+
=
\begin{bmatrix}
\Delta u(k) &
\Delta u(k+1) &
\cdots &
\Delta u(k+h-1)
\end{bmatrix}^{T}.
\end{align*}

La matriz dinámica es triangular inferior:

\begin{align*}
G
=
\begin{bmatrix}
g_1 & 0 & \cdots & 0 \\
g_2 & g_1 & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
g_h & g_{h-1} & \cdots & g_1
\end{bmatrix}.
\end{align*}

Para la respuesta libre se utiliza

\begin{align*}
\mathbf f
=
F\Delta\mathbf u^-
+
\mathbf 1\,y(k),
\end{align*}

donde

\begin{align*}
\Delta\mathbf u^-
=
\begin{bmatrix}
\Delta u(k-1) &
\Delta u(k-2) &
\cdots &
\Delta u(k-N_t+1)
\end{bmatrix}^{T}.
\end{align*}

Los elementos de $F$ se obtienen mediante

\begin{align*}
F_{j,i}
=
g_{i+j}-g_i,
\end{align*}

adoptando

\begin{align*}
g_m=g_{N_t},
\qquad m>N_t.
\end{align*}

In [ ]:
def create_dmc_matrices(gi, horizon):
    """Construye las matrices G y F del predictor DMC SISO."""
    gi = np.asarray(gi, dtype=float).reshape(-1)
    Nt = len(gi)

    if horizon > Nt:
        raise ValueError("Se requiere horizon <= Nt para construir este predictor.")

    # Matriz de respuesta forzada
    G = np.zeros((horizon, horizon))
    for row in range(horizon):
        for col in range(row + 1):
            G[row, col] = gi[row - col]

    # Matriz de respuesta libre.
    # El último incremento pasado no es necesario porque, con la extensión
    # g_m = g_Nt para m > Nt, su contribución es nula.
    F = np.zeros((horizon, Nt - 1))

    def g(index_1based):
        if index_1based <= Nt:
            return gi[index_1based - 1]
        return gi[-1]

    for j in range(1, horizon + 1):
        for i in range(1, Nt):
            F[j - 1, i - 1] = g(i + j) - g(i)

    return G, F


horizon = 10
G, F = create_dmc_matrices(gi, horizon)

print("Dimensión de G:", G.shape)
print("Dimensión de F:", F.shape)
print()
print("G =")
print(G)

### Preguntas para dinamizar — predictor vectorial DMC

> **Pregunta 21.** ¿Por qué la matriz $G$ es triangular inferior?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Por causalidad. Un incremento futuro solo puede afectar a la salida desde el instante en que se aplica en adelante.

</details>

> **Pregunta 22.** ¿Qué representa la primera columna de $G$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El efecto del incremento que aplicamos ahora, $\Delta u(k)$, sobre todas las predicciones futuras:

\begin{align*}
[g_1,g_2,\ldots,g_h]^T.
\end{align*}

</details>

> **Pregunta 23.** ¿Qué representa la última columna?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El efecto del último incremento incluido en el horizonte, $\Delta u(k+h-1)$, que solo puede afectar a la última predicción.

</details>

> **Pregunta 24.** ¿Cuál es la diferencia conceptual entre $\Delta\mathbf u^+$ y $\Delta\mathbf u^-$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

$\Delta\mathbf u^+$ contiene incrementos futuros que el optimizador puede decidir. $\Delta\mathbf u^-$ contiene incrementos ya aplicados que no pueden modificarse, pero cuyo efecto aún permanece en la planta.

</details>

In [ ]:
# Experimento en directo: visualizar la estructura de G

plt.figure(figsize=(6, 5))
plt.imshow(np.abs(G), aspect="auto")
plt.colorbar(label="|G[i,j]|")
plt.xlabel("Incremento futuro j")
plt.ylabel("Predicción futura i")
plt.title("Estructura de la matriz dinámica G")
plt.show()

print("Primera columna de G:")
print(G[:, 0])

print("\nÚltima columna de G:")
print(G[:, -1])

### Respuesta libre y memoria de incrementos pasados

> **Pregunta 25.** Si fijamos todos los incrementos futuros a cero, $\Delta\mathbf u^+=0$, ¿qué predicción queda?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Queda únicamente

\begin{align*}
\hat{\mathbf y}=\mathbf f,
\end{align*}

la respuesta libre.

</details>

> **Pregunta 26.** Si $\Delta u(k)=0$ a partir de ahora, ¿significa que $u(k)=0$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Significa que la entrada deja de cambiar:

\begin{align*}
u(k)=u(k-1).
\end{align*}

Puede mantenerse una entrada constante distinta de cero.

</details>

> **Pregunta 27.** ¿Puede haber dos respuestas libres distintas con el mismo valor actual $y(k)$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Sí. Si las historias de incrementos pasados son diferentes, el término

\begin{align*}
F\Delta\mathbf u^-
\end{align*}

también será diferente aunque la salida actual coincida.

</details>

In [ ]:
# Experimento conceptual: misma salida actual, distinta historia de incrementos

yk_demo = 0.5

du_past_A = np.zeros(Nt - 1)
du_past_B = np.zeros(Nt - 1)
du_past_B[0] = 0.5

f_A = F @ du_past_A + np.ones(horizon) * yk_demo
f_B = F @ du_past_B + np.ones(horizon) * yk_demo

plt.figure(figsize=(8, 4))
plt.plot(np.arange(1, horizon + 1), f_A, "o-", label="Historia A")
plt.plot(np.arange(1, horizon + 1), f_B, "s-", label="Historia B")
plt.axhline(yk_demo, linestyle="--", label="y(k)")
plt.xlabel("Paso futuro")
plt.ylabel("Respuesta libre")
plt.title("Misma salida actual, distinta historia de incrementos")
plt.grid(True)
plt.legend()
plt.show()

## 7. Función objetivo y ley de control

El DMC sin restricciones minimiza

\begin{align*}
J
=
(\hat{\mathbf y}-\mathbf w)^T
(\hat{\mathbf y}-\mathbf w)
+
\lambda
{\Delta\mathbf u^+}^T
\Delta\mathbf u^+.
\end{align*}

Sustituyendo

\begin{align*}
\hat{\mathbf y}
=
G\Delta\mathbf u^+
+
\mathbf f,
\end{align*}

la solución analítica es

\begin{align*}
\Delta\mathbf u^*
=
\left(
G^T G+\lambda I
\right)^{-1}
G^T
(\mathbf w-\mathbf f).
\end{align*}

Como solo se aplica el primer elemento del vector óptimo,

\begin{align*}
\boxed{
\Delta u(k)
=
K_1(\mathbf w-\mathbf f)
}.
\end{align*}

La acción de control absoluta se actualiza mediante

\begin{align*}
u(k)=u(k-1)+\Delta u(k).
\end{align*}

### Valores de partida

Se emplean como referencia

\begin{align*}
h&=10, \\
\lambda&=50,
\end{align*}

dejando las modificaciones del script para los experimentos posteriores.

In [ ]:
lam = 50.0

K = np.linalg.solve(
    G.T @ G + lam * np.eye(horizon),
    G.T
)

K1 = K[0, :]

print("K1 =")
print(K1)

### Preguntas para dinamizar — función objetivo y ley de control

> **Pregunta 28.** Si $\lambda=0$, ¿qué único objetivo queda?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Solo se penaliza el error de seguimiento. El optimizador no penaliza explícitamente cambios grandes de la entrada.

</details>

> **Pregunta 29.** ¿Qué sucede cualitativamente al aumentar $\lambda$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Se penalizan más los incrementos. El control tiende a ser más suave y el seguimiento más lento.

</details>

> **Pregunta 30.** ¿Penalizar $\Delta u$ es lo mismo que penalizar $u$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Penalizar $\Delta u$ castiga los cambios de entrada, pero permite mantener sin coste incremental una entrada constante distinta de cero.

</details>

> **Pregunta 31.** ¿Qué ventaja práctica tiene penalizar $\Delta u$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Favorece movimientos suaves del actuador, reduce cambios bruscos y permite mantener el valor de entrada necesario en régimen estacionario.

</details>

> **Pregunta 32.** ¿Por qué usamos `np.linalg.solve()` en vez de calcular explícitamente una inversa?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque resolver directamente el sistema lineal es numéricamente preferible y evita calcular una inversa innecesaria.

</details>

In [ ]:
# Experimento en directo: efecto de lambda sobre K1 y el condicionamiento

lambda_demo = [0.0, 0.1, 1.0, 5.0, 20.0, 50.0, 200.0]

for lam_test in lambda_demo:
    M = G.T @ G + lam_test * np.eye(horizon)
    try:
        K_test = np.linalg.solve(M, G.T)
        K1_test = K_test[0, :]
        print(
            f"lambda={lam_test:7.3g}  "
            f"||K1||2={np.linalg.norm(K1_test):9.4f}  "
            f"cond={np.linalg.cond(M):12.4e}"
        )
    except np.linalg.LinAlgError:
        print(f"lambda={lam_test:7.3g} -> matriz singular")

> **Pregunta 33.** Si calculamos una secuencia completa de incrementos óptimos, ¿por qué aplicamos solo el primero?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque en la siguiente muestra medimos de nuevo la planta y resolvemos otra vez el problema con información actualizada. Ese es el principio de horizonte deslizante.

</details>

## 8. Simulación DMC

El bucle de horizonte deslizante realiza en cada instante:

1. medida de $y(k)$;
2. cálculo de la respuesta libre $\mathbf f$;
3. construcción de la referencia futura $\mathbf w$;
4. cálculo de $\Delta u(k)$;
5. actualización de $u(k)$;
6. aplicación de $u(k)$ a la planta;
7. actualización del buffer de incrementos pasados;
8. repetición en el siguiente instante.

En la siguiente celda se implementan las funciones que se emplearán para la simulación DMC.

In [ ]:
def future_reference(ref, k, horizon):
    """Construye [r(k+1), ..., r(k+h)], manteniendo el último valor al final."""
    w = np.empty(horizon)

    for j in range(1, horizon + 1):
        idx = min(k + j, len(ref) - 1)
        w[j - 1] = ref[idx]

    return w


def simulate_dmc(Ad, Bd, Cd, Dd, gi, horizon, lam, ref):
    """Simulación de un controlador DMC SISO sin restricciones."""
    G, F = create_dmc_matrices(gi, horizon)

    K = np.linalg.solve(
        G.T @ G + lam * np.eye(horizon),
        G.T
    )
    K1 = K[0, :]

    Nt = len(gi)
    N = len(ref) - 1

    y = np.zeros(N + 1)
    u = np.zeros(N)
    du = np.zeros(N)

    x = 0.0
    u_prev = 0.0

    # [Delta u(k-1), Delta u(k-2), ..., Delta u(k-Nt+1)]
    du_past = np.zeros(Nt - 1)

    for k in range(N):
        y[k] = Cd * x + Dd * u_prev

        # Respuesta libre
        f = F @ du_past + np.ones(horizon) * y[k]

        # Referencia futura
        w = future_reference(ref, k, horizon)

        # Primer incremento de la secuencia óptima
        duk = float(K1 @ (w - f))
        uk = u_prev + duk

        du[k] = duk
        u[k] = uk

        # Evolución de la planta
        x = Ad * x + Bd * uk

        # Actualización del buffer
        if len(du_past) > 1:
            du_past[1:] = du_past[:-1]
        if len(du_past) > 0:
            du_past[0] = duk

        u_prev = uk

    y[N] = Cd * x + Dd * u_prev

    return y, u, du

### Preguntas para dinamizar — una iteración DMC

> **Pregunta 34.** ¿Qué variable actualiza realmente el optimizador: $u(k)$ o $\Delta u(k)$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El optimizador calcula $\Delta u(k)$. Después se reconstruye la entrada absoluta:

\begin{align*}
u(k)=u(k-1)+\Delta u(k).
\end{align*}

</details>

> **Pregunta 35.** ¿Por qué hay que guardar un buffer de incrementos pasados?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque un cambio de entrada aplicado anteriormente sigue influyendo en la salida durante muchas muestras. El buffer permite incluir ese efecto en la respuesta libre.

</details>

> **Pregunta 36.** ¿Qué diferencia habría entre aplicar toda la secuencia calculada y recalcular muestra a muestra?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Aplicar toda la secuencia sería esencialmente lazo abierto durante el horizonte. Recalcular permite corregir perturbaciones, errores de modelo y desviaciones de la predicción.

</details>

## 9. Referencia de la práctica

Se genera la siguiente referencia para la práctica DMC:

\begin{align*}
0
\;\rightarrow\;
1
\;\rightarrow\;
-1.
\end{align*}

Se genera un vector de $N$ elementos donde $N=1000$.

In [ ]:
N = 1000

ref = np.zeros(N + 1)
n1 = N // 3
n2 = 2 * N // 3

ref[n1:n2] = 1.0
ref[n2:] = -1.0

y_dmc, u_dmc, du_dmc = simulate_dmc(
    Ad, Bd, Cd, Dd,
    gi=gi,
    horizon=horizon,
    lam=lam,
    ref=ref
)

k_y = np.arange(N + 1)
k_u = np.arange(N)

plt.figure(figsize=(10, 4))
plt.plot(k_y, ref, label="Referencia")
plt.plot(k_y, y_dmc, label="Salida")
plt.xlabel("k")
plt.ylabel("y(k)")
plt.title("Control DMC")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.step(k_u, u_dmc, where="post")
plt.xlabel("k")
plt.ylabel("u(k)")
plt.title("Acción de control")
plt.grid(True)
plt.show()

plt.figure(figsize=(10, 4))
plt.step(k_u, du_dmc, where="post")
plt.xlabel("k")
plt.ylabel("Delta u(k)")
plt.title("Incrementos de la acción de control")
plt.grid(True)
plt.show()

### Referencia futura y anticipación

> **Pregunta 37.** Si la referencia cambiará dentro de 5 muestras y $h=10$, ¿puede DMC actuar antes del cambio?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Sí. El nuevo valor ya aparece dentro del vector de referencia futura $\mathbf w$ y puede influir en la optimización actual.

</details>

> **Pregunta 38.** ¿Hasta cuántas muestras antes puede ver un cambio con $h=10$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Hasta 10 muestras hacia delante.

</details>

> **Pregunta 39.** ¿Qué ocurriría si repitiésemos simplemente el valor actual $r(k)$ en todo el horizonte?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Se perdería la anticipación de cambios conocidos de consigna. El controlador seguiría prediciendo la planta, pero reaccionaría al cambio de referencia solo cuando este fuese actual.

</details>

In [ ]:
# Experimento en directo: DMC con y sin preview de referencia

def held_reference(ref, k, horizon):
    return np.full(horizon, ref[min(k, len(ref)-1)])

def simulate_dmc_no_preview(Ad, Bd, Cd, Dd, gi, horizon, lam, ref):
    G_np, F_np = create_dmc_matrices(gi, horizon)
    K_np = np.linalg.solve(G_np.T @ G_np + lam * np.eye(horizon), G_np.T)
    K1_np = K_np[0, :]

    Nt_np = len(gi)
    N_np = len(ref) - 1

    y = np.zeros(N_np + 1)
    u = np.zeros(N_np)
    du = np.zeros(N_np)

    x = 0.0
    u_prev = 0.0
    du_past = np.zeros(Nt_np - 1)

    for k in range(N_np):
        y[k] = Cd * x + Dd * u_prev
        f = F_np @ du_past + np.ones(horizon) * y[k]
        w = held_reference(ref, k, horizon)

        duk = float(K1_np @ (w - f))
        uk = u_prev + duk

        du[k] = duk
        u[k] = uk
        x = Ad * x + Bd * uk

        if len(du_past) > 1:
            du_past[1:] = du_past[:-1]
        if len(du_past) > 0:
            du_past[0] = duk

        u_prev = uk

    y[N_np] = Cd * x + Dd * u_prev
    return y, u, du

y_preview, u_preview, _ = simulate_dmc(
    Ad, Bd, Cd, Dd, gi, horizon, lam, ref
)
y_no_preview, u_no_preview, _ = simulate_dmc_no_preview(
    Ad, Bd, Cd, Dd, gi, horizon, lam, ref
)

k0 = n1
window = 30
sl_y = slice(k0-window, k0+window+1)
sl_u = slice(k0-window, k0+window)

plt.figure(figsize=(10, 4))
plt.plot(k_y[sl_y], ref[sl_y], label="Referencia")
plt.plot(k_y[sl_y], y_preview[sl_y], label="DMC con preview")
plt.plot(k_y[sl_y], y_no_preview[sl_y], label="DMC sin preview")
plt.axvline(k0, linestyle="--", label="Cambio de referencia")
plt.xlabel("k")
plt.ylabel("y(k)")
plt.title("Anticipación gracias a la referencia futura")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.step(k_u[sl_u], u_preview[sl_u], where="post", label="u con preview")
plt.step(k_u[sl_u], u_no_preview[sl_u], where="post", label="u sin preview")
plt.axvline(k0, linestyle="--", label="Cambio de referencia")
plt.xlabel("k")
plt.ylabel("u(k)")
plt.title("Acción de control alrededor del cambio de referencia")
plt.grid(True)
plt.legend()
plt.show()

> **Pregunta 40.** ¿Por qué puede empezar a cambiar $u(k)$ antes del salto de referencia?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque el DMC sabe que el salto llegará dentro del horizonte y utiliza el modelo para distribuir anticipadamente la acción de control.

</details>

## 10. Análisis del error estacionario

Una diferencia importante respecto al MAC de la práctica anterior es que DMC penaliza los **incrementos** $\Delta u$ y no directamente el valor absoluto $u$.

Cuando se alcanza el régimen estacionario puede mantenerse

\begin{align*}
\Delta u(k)=0
\end{align*}

aunque

\begin{align*}
u(k)\neq 0.
\end{align*}

Esto permite mantener la acción necesaria para compensar la dinámica estacionaria de la planta sin seguir penalizando continuamente su valor absoluto.

Como el DMC conoce la referencia futura dentro del horizonte, empieza a anticipar el siguiente cambio de consigna durante las últimas $h$ muestras de cada tramo. Por ello, al estimar el error estacionario de un tramo se excluye esa ventana final de anticipación.

In [ ]:
def mean_error_before_preview(ref, y, start, end, horizon, n_last=50, is_final=False):
    # En tramos no finales se excluyen las últimas h muestras porque el
    # controlador ya ve el siguiente cambio de referencia.
    effective_end = end if is_final else max(start, end - horizon)
    a = max(start, effective_end - n_last)
    return float(np.mean(ref[a:effective_end] - y[a:effective_end]))

segments = [
    (n1, n2, 1.0, False),
    (n2, N, -1.0, True),
]

for start, end, r_value, is_final in segments:
    e_ss = mean_error_before_preview(
        ref, y_dmc, start, end,
        horizon=horizon,
        is_final=is_final
    )
    print(f"Referencia {r_value:+.3f}: error medio final ≈ {e_ss:+.8f}")

### Preguntas para dinamizar — régimen estacionario

> **Pregunta 41.** ¿Presenta DMC un error estacionario apreciable en esta práctica ideal?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Debe ser muy pequeño, porque el modelo es exacto y el controlador puede mantener una entrada constante sin penalización incremental una vez alcanzado el régimen.

</details>

> **Pregunta 42.** ¿Por qué puede mantenerse $u(k)\neq0$ con coste incremental nulo?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque en estacionario

\begin{align*}
\Delta u(k)=0
\end{align*}

aunque $u(k)$ sea distinto de cero.

</details>

> **Pregunta 43.** ¿Por qué este comportamiento difiere del MAC anterior, que penalizaba $u$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

En MAC una entrada constante no nula sigue teniendo coste. En DMC, una entrada constante tiene $\Delta u=0$ y deja de penalizarse.

</details>

> **Pregunta 44.** Para mantener $y=1$ en estacionario, ¿qué entrada necesita idealmente esta planta?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Como

\begin{align*}
q=K_t h,
\end{align*}

con $K_t=0.5$ y $h=1$,

\begin{align*}
q_{\mathrm{ss}}=0.5.
\end{align*}

</details>

> **Pregunta 45.** ¿Qué ocurriría con el error estacionario si existiese un error permanente de ganancia en el modelo?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El resultado podría dejar de ser offset-free. La ausencia de error observada aquí depende de la formulación y de que el modelo utilizado represente adecuadamente la planta.

</details>

In [ ]:
# Experimento en directo: comprobar u y Delta u en régimen estacionario

a = n2 - horizon - 50
b = n2 - horizon

print("Tramo r=1 antes de la ventana de preview:")
print(f"y medio      = {np.mean(y_dmc[a:b]):.8f}")
print(f"u medio      = {np.mean(u_dmc[a:b]):.8f}")
print(f"Delta u medio= {np.mean(du_dmc[a:b]):.8f}")
print("Valor ideal de u para y=1:", Kt * 1.0)

## 11. Experimento A — Efecto del horizonte de predicción $h$

Se propone comparar

\begin{align*}
h=5,\qquad h=10,\qquad h=20.
\end{align*}

Debe mantenerse $h\leq N_t$.

Un horizonte mayor permite considerar una evolución futura más larga, pero también aumenta el tamaño del problema y no garantiza por sí solo una mejora ilimitada del comportamiento.

In [ ]:
h_values = [5, 10, 20]

plt.figure(figsize=(10, 4))
plt.plot(k_y, ref, label="Referencia")

for h_test in h_values:
    y_test, _, _ = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi,
        horizon=h_test,
        lam=lam,
        ref=ref
    )
    plt.plot(k_y, y_test, label=f"h={h_test}")

plt.xlabel("k")
plt.ylabel("y(k)")
plt.title("Efecto del horizonte de predicción")
plt.grid(True)
plt.legend()
plt.show()

### Preguntas para dinamizar — efecto del horizonte $h$

> **Pregunta 46.** Antes de ejecutar, ¿qué esperáis que cambie al pasar de $h=5$ a $h=20$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El controlador considera consecuencias más lejanas y dispone de más capacidad de anticipación. También aumenta el tamaño del problema.

</details>

> **Pregunta 47.** ¿Un horizonte mayor garantiza siempre mejor control?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Una vez cubierta la dinámica relevante, aumentar $h$ puede aportar poca mejora y aumentar el coste computacional.

</details>

> **Pregunta 48.** ¿Qué significado tendría $h=0$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No existiría un futuro sobre el que predecir ni optimizar. La formulación predictiva dejaría de tener sentido.

</details>

> **Pregunta 49.** Si la planta tuviese un tiempo muerto mayor que $h$, ¿qué problema aparecería?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Dentro del horizonte el controlador apenas vería efecto de sus incrementos actuales sobre la salida. El horizonte debería extenderse más allá del tiempo muerto.

</details>

In [ ]:
# Cuantificación del efecto de h

print("h       max|du|        error_ss(r=1)")
print("------------------------------------")

for h_test in h_values:
    y_test, _, du_test = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi,
        horizon=h_test,
        lam=lam,
        ref=ref
    )
    e_ss = mean_error_before_preview(
        ref, y_test, n1, n2,
        horizon=h_test, is_final=False
    )
    print(
        f"{h_test:2d}      "
        f"{np.max(np.abs(du_test)):10.6f}      "
        f"{e_ss:+14.8f}"
    )

> **Pregunta 50.** ¿A partir de qué valor de $h$ las mejoras parecen pequeñas para esta planta?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Debe decidirse observando las curvas y las métricas ejecutadas. No existe un valor universal.

</details>

> **Pregunta 51.** ¿Qué relación debe respetarse entre $h$ y $N_t$ en este notebook?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Se requiere

\begin{align*}
h\leq N_t.
\end{align*}

El modelo debe contener suficientes coeficientes para construir el predictor a lo largo del horizonte.

</details>

## 12. Experimento B — Efecto de $\lambda$

El parámetro $\lambda$ pondera los incrementos de la acción de control:

\begin{align*}
J
=
\|\hat{\mathbf y}-\mathbf w\|_2^2
+
\lambda
\|\Delta\mathbf u^+\|_2^2.
\end{align*}

Una $\lambda$ elevada penaliza cambios rápidos o grandes de $u$.

In [ ]:
lambda_values = [5.0, 20.0, 50.0, 200.0]

plt.figure(figsize=(10, 4))
plt.plot(k_y, ref, label="Referencia")

for lam_test in lambda_values:
    y_test, _, _ = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi,
        horizon=horizon,
        lam=lam_test,
        ref=ref
    )
    plt.plot(k_y, y_test, label=f"lambda={lam_test}")

plt.xlabel("k")
plt.ylabel("y(k)")
plt.title("Efecto de lambda sobre la salida")
plt.grid(True)
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))

for lam_test in lambda_values:
    _, _, du_test = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi,
        horizon=horizon,
        lam=lam_test,
        ref=ref
    )
    plt.step(k_u, du_test, where="post", label=f"lambda={lam_test}")

plt.xlabel("k")
plt.ylabel("Delta u(k)")
plt.title("Efecto de lambda sobre los incrementos de control")
plt.grid(True)
plt.legend()
plt.show()

### Preguntas para dinamizar — efecto de $\lambda$

> **Pregunta 52.** ¿Qué curva esperáis que presente mayores picos de $\Delta u$: $\lambda=5$ o $\lambda=200$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

La de $\lambda=5$, porque los incrementos están menos penalizados.

</details>

> **Pregunta 53.** ¿Qué efecto tendrá aumentar $\lambda$ sobre la velocidad de seguimiento?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

El seguimiento será normalmente más lento, ya que el controlador reparte la corrección en cambios de entrada más pequeños.

</details>

> **Pregunta 54.** ¿Aumentar $\lambda$ garantiza respetar una restricción dura sobre $\Delta u$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Solo penaliza el incremento. Para garantizar

\begin{align*}
\Delta u_{\min}\leq\Delta u(k)\leq\Delta u_{\max}
\end{align*}

se necesita una formulación con restricciones explícitas.

</details>

> **Pregunta 55.** ¿Puede reducirse $\lambda$ hasta cero sin ninguna consecuencia?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No necesariamente. La acción puede volverse muy agresiva y puede empeorar el condicionamiento numérico de $G^TG$.

</details>

In [ ]:
# Cuantificación de lambda

print("lambda      max|du|       max|u|        error_ss(r=1)")
print("------------------------------------------------------")

for lam_test in lambda_values:
    y_test, u_test, du_test = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi,
        horizon=horizon,
        lam=lam_test,
        ref=ref
    )
    e_ss = mean_error_before_preview(
        ref, y_test, n1, n2,
        horizon=horizon, is_final=False
    )
    print(
        f"{lam_test:7.1f}    "
        f"{np.max(np.abs(du_test)):10.6f}    "
        f"{np.max(np.abs(u_test)):10.6f}    "
        f"{e_ss:+14.8f}"
    )

> **Pregunta 56.** ¿Cuál es el compromiso principal al seleccionar $\lambda$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Rapidez de seguimiento frente a suavidad de los movimientos del actuador.

</details>

> **Pregunta 57.** ¿Por qué no existe un $\lambda$ universalmente óptimo?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque depende de la dinámica, de los límites del actuador y de las prioridades de operación.

</details>

## 13. Experimento C — Efecto del truncamiento $N_t$

El DMC supone que a partir de $N_t$ la respuesta escalón ha alcanzado aproximadamente su valor estacionario:

\begin{align*}
g_i\approx g_{N_t},
\qquad i>N_t.
\end{align*}

Un $N_t$ demasiado pequeño introduce un error sistemático en el predictor.

In [ ]:
Nt_values = [20, 40, 60, 80]

plt.figure(figsize=(10, 4))
plt.plot(k_y, ref, label="Referencia")

for Nt_test in Nt_values:
    gi_test = g_full[:Nt_test]

    y_test, _, _ = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi_test,
        horizon=min(horizon, Nt_test),
        lam=lam,
        ref=ref
    )

    plt.plot(k_y, y_test, label=f"Nt={Nt_test}")

plt.xlabel("k")
plt.ylabel("y(k)")
plt.title("Efecto del truncamiento del modelo escalón")
plt.grid(True)
plt.legend()
plt.show()

print("Error asociado al valor final de la respuesta escalón:")
for Nt_test in Nt_values:
    print(
        f"Nt={Nt_test:2d}: "
        f"{100*truncation_error_step(g_full, Nt_test, g_inf):7.4f} %"
    )

### Preguntas para dinamizar — $N_t$ en lazo cerrado

> **Pregunta 58.** Al modificar $N_t$, ¿cambiamos la planta o solo el modelo del controlador?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Solo el modelo utilizado por DMC. La planta real simulada permanece idéntica.

</details>

> **Pregunta 59.** ¿Por qué un $N_t$ corto puede alterar la respuesta libre?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque el controlador supone que la respuesta escalón ya se ha estabilizado y reemplaza prematuramente la cola por $g_{N_t}$.

</details>

> **Pregunta 60.** ¿Qué criterio usaríais para decidir entre $N_t=20$ y $N_t=60$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Compararía el error de truncamiento y la mejora real en las curvas de lazo cerrado frente al incremento de memoria y cálculo.

</details>

In [ ]:
# Cuantificación del efecto de Nt en lazo cerrado

print("Nt      error trunc.[%]      max|du|      error_ss(r=1)")
print("-------------------------------------------------------")

for Nt_test in Nt_values:
    gi_test = g_full[:Nt_test]
    h_test = min(horizon, Nt_test)

    y_test, _, du_test = simulate_dmc(
        Ad, Bd, Cd, Dd,
        gi=gi_test,
        horizon=h_test,
        lam=lam,
        ref=ref
    )

    e_ss = mean_error_before_preview(
        ref, y_test, n1, n2,
        horizon=h_test, is_final=False
    )

    print(
        f"{Nt_test:2d}      "
        f"{100*truncation_error_step(g_full, Nt_test, g_inf):12.6f}      "
        f"{np.max(np.abs(du_test)):9.6f}      "
        f"{e_ss:+14.8f}"
    )

> **Pregunta 61.** ¿Aumentar $N_t$ indefinidamente mejora necesariamente el control?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

No. Cuando el error de truncamiento ya es despreciable, añadir más coeficientes apenas aporta información nueva.

</details>

> **Pregunta 62.** Si reducimos $T_d$, ¿es probable que necesitemos aumentar $N_t$ para representar la misma duración física?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Sí, porque

\begin{align*}
T_{\mathrm{mem}}\simeq N_tT_d.
\end{align*}

</details>

## 14. Comparación conceptual MAC — DMC

Las dos primeras prácticas utilizan modelos de convolución, pero toman decisiones distintas.

### MAC

MAC representa la planta mediante una respuesta impulsional truncada:

\begin{align*}
y(k)
=
\sum_{i=1}^{N_t}
h_i u(k-i),
\end{align*}

y en la formulación estudiada penaliza el valor de la entrada:

\begin{align*}
J_{\mathrm{MAC}}
=
\|\hat{\mathbf y}-\mathbf w\|_2^2
+
\lambda
\|\mathbf u^+\|_2^2.
\end{align*}

### DMC

DMC representa la planta mediante una respuesta escalón y utiliza incrementos de control:

\begin{align*}
y(k)
=
\sum_{i=1}^{N_t}
g_i\Delta u(k-i),
\end{align*}

con

\begin{align*}
J_{\mathrm{DMC}}
=
\|\hat{\mathbf y}-\mathbf w\|_2^2
+
\lambda
\|\Delta\mathbf u^+\|_2^2.
\end{align*}

### Actividad

Explique por qué esta diferencia ayuda a entender el distinto comportamiento estacionario observado en las dos prácticas.

### Preguntas para dinamizar — comparación MAC y DMC

> **Pregunta 63.** ¿Cuál es la diferencia fundamental entre los modelos de las dos prácticas?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

MAC utiliza la respuesta impulsional $h_i$ y DMC la respuesta escalón $g_i$.

</details>

> **Pregunta 64.** ¿Qué variable futura optimiza cada formulación estudiada?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

MAC optimiza directamente $\mathbf u^+$, mientras que DMC optimiza $\Delta\mathbf u^+$.

</details>

> **Pregunta 65.** ¿Por qué esta diferencia ayuda a explicar el distinto error estacionario observado?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

En MAC una entrada constante no nula sigue siendo penalizada. En DMC, una vez que la entrada alcanza su valor estacionario,

\begin{align*}
\Delta u=0,
\end{align*}

y deja de existir penalización incremental.

</details>

> **Pregunta 66.** ¿Puede obtenerse la respuesta impulsional a partir de la escalón?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Sí:

\begin{align*}
h_i=g_i-g_{i-1}.
\end{align*}

Ambos modelos contienen información dinámica estrechamente relacionada.

</details>

## 15. Actividad final

Seleccione razonadamente los parámetros

\begin{align*}
T_d,\qquad
N_t,\qquad
h,\qquad
\lambda
\end{align*}

para la planta estudiada.

La justificación debe considerar:

- resolución temporal;
- calidad del modelo escalón;
- rapidez de seguimiento;
- suavidad de la acción de control;
- error estacionario;
- coste computacional.

### Entregable propuesto

Incluya:

1. valores finales de los parámetros;
2. gráfica de referencia y salida;
3. gráfica de $u(k)$;
4. gráfica de $\Delta u(k)$;
5. cuantificación del error estacionario;
6. cuantificación del error de truncamiento;
7. comparación breve con la práctica MAC.

### Preguntas de síntesis antes de seleccionar los parámetros finales

> **Pregunta 67.** ¿Qué parámetros describen principalmente el modelo y cuáles sintonizan el controlador?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

- $T_d$ y $N_t$: representación temporal y longitud del modelo escalón.
- $h$: cantidad de futuro utilizada en la optimización.
- $\lambda$: compromiso entre seguimiento y cambios de la acción de control.

</details>

> **Pregunta 68.** ¿En qué orden tendría sentido seleccionarlos?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Una secuencia razonable es:

1. elegir $T_d$ según la dinámica;
2. obtener la respuesta escalón;
3. seleccionar $N_t$ con un error de truncamiento aceptable;
4. seleccionar $h$ para cubrir la dinámica relevante;
5. ajustar $\lambda$ según rapidez y suavidad del actuador.

</details>

> **Pregunta 69.** Si el seguimiento es malo, ¿por qué no deberíamos modificar inmediatamente $\lambda$?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

Porque el problema puede estar en el modelo: periodo de muestreo inadecuado, truncamiento demasiado corto o horizonte insuficiente.

</details>

> **Pregunta 70.** Si tuvierais que resumir DMC en una sola frase, ¿cuál sería?

<details>
<summary><b>Respuesta / comentario docente</b></summary>

DMC utiliza la respuesta escalón del proceso para predecir la salida futura y calcula en cada muestra los incrementos de control que minimizan un compromiso entre seguimiento y suavidad, aplicando solo el primer incremento y repitiendo la optimización en horizonte deslizante.

</details>